# ReviewPulse — Exploratory Data Analysis

Multi-Domain Sentiment Dataset (Blitzer et al., 2007)  
8,000 labelled Amazon reviews across 4 domains: Books, DVDs, Electronics, Kitchen & Housewares.

**Goals:**
- Understand class and domain balance
- Inspect rating distributions
- Validate outlier removal thresholds (min/max word count)
- Audit label quality
- Inform ethics discussion for the report

In [ ]:
import sys
sys.path.insert(0, '..')

import matplotlib.pyplot as plt
import pandas as pd

from src.parser import load_all_domains
from src.features import (
    class_balance,
    domain_balance,
    label_audit_summary,
    length_stats,
    plot_domain_balance,
    plot_length_distribution,
    rating_distribution,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

## 1. Load raw data

In [ ]:
df = load_all_domains()
print(f"Total reviews: {len(df):,}")
df.head(3)

## 2. Class balance

A well-balanced dataset means accuracy is a reliable metric and the model won't be biased toward one class.

In [ ]:
class_balance(df)

## 3. Domain balance

In [ ]:
print(domain_balance(df))
plot_domain_balance(df)

## 4. Rating distribution

We use the **filename** as the primary label, not the star rating.  
Retaining ratings allows us to audit label quality and flag conflicts.

In [ ]:
rating_distribution(df)

In [ ]:
# Rating distribution per label
df.groupby(['label', 'rating']).size().unstack(fill_value=0).rename(index={0: 'negative', 1: 'positive'})

## 5. Review length distribution

Used to validate the outlier removal thresholds (`min_words=10`, `max_words=500`).  
Adjust these values if the distribution warrants different cutoffs.

In [ ]:
length_stats(df)

In [ ]:
plot_length_distribution(df, min_words=10, max_words=500)

In [ ]:
# How many reviews fall outside the thresholds?
wc = df['text'].str.split().str.len()
n_short = (wc < 10).sum()
n_long  = (wc > 500).sum()
print(f"Too short (< 10 words):  {n_short} ({n_short/len(df)*100:.1f}%)")
print(f"Too long  (> 500 words): {n_long}  ({n_long/len(df)*100:.1f}%)")
print(f"Retained after removal:  {len(df) - n_short - n_long}")

## 6. Label audit

Check for ambiguous (3-star) reviews and rating/filename conflicts.

In [ ]:
label_audit_summary(df)

## 7. Sample reviews

Spot-check both classes to get a feel for the language.

In [ ]:
print("=== Positive sample ===")
print(df[df['label'] == 1]['text'].iloc[0][:500])
print("\n=== Negative sample ===")
print(df[df['label'] == 0]['text'].iloc[0][:500])

## 8. Conclusions

| Finding | Implication |
|---|---|
| Dataset is perfectly balanced (50/50) | Accuracy is a reliable metric; no class weighting needed |
| 1,000 reviews per domain per class | No domain dominates training |
| 0 ambiguous (3-star) reviews | Dataset owners already excluded neutral reviews |
| 0 rating/filename conflicts | Label quality is high |
| Most reviews are 50–300 words | Thresholds min=10, max=500 are appropriate |
| ~3.5% of reviews are outliers | Small enough to drop without significant data loss |

**Ethics note:** The dataset covers only four Amazon product domains. The trained model may not generalise to other sentiment contexts (social media, healthcare, finance). Binary positive/negative classification also cannot capture neutral, mixed, or sarcastic sentiment — a limitation to be documented in the report.